In [74]:
from qdrant_client import QdrantClient

client = QdrantClient(host="localhost", port=6333)

collection_name = "annual_report_debug"

try:
    client.delete_collection(collection_name)
    print("Old collection deleted")
except:
    print("Collection did not exist")

Old collection deleted


In [75]:
import os
from dotenv import load_dotenv

load_dotenv()

print("GROQ KEY loaded:", os.getenv("GROQ_API_KEY")[:10])

GROQ KEY loaded: gsk_YRzXDf


In [76]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "../data/annual_reports/HSBC_ANNUAL_REPORT.pdf"

loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

print("Total pages:", len(docs))

print("\nExample page text:\n")
print(docs[10].page_content[:500])

Total pages: 372

Example page text:

Our strategy
In 2025, we continued to implement our strategy that supports our ambition to be the most 
trusted bank globally, putting customers at the heart of everything we do.
A growing, high-returning HSBC
Our strategic priorities remain clear: we aim to 
drive customer-centricity, deliver focused 
sustainable growth, and be simple and more 
agile.
We are intensely focused on our customers. 
The depth and quality of our customer 
relationships and our ability to connect 
customers globally h


In [77]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1024,
    chunk_overlap=256
)

chunks = text_splitter.split_documents(docs)

print("Total chunks:", len(chunks))

print("\nExample chunk:\n")
print(chunks[100].page_content[:400])

Total chunks: 2571

Example chunk:

Income statement results
2025 compared with 2024 
Movement in reported profit before tax compared with 2024
2025 2024 2023 2025 vs 2024
of which strategic 
transactions1
Reported results $m $m $m $m % $m
Revenue  68,274  65,854  66,058  2,420  4  (1,936) 
–  of which: net interest income  34,794  32,733  35,796  2,061  6  (1,628) 
ECL  (3,850)  (3,414)  (3,447)  (436)  (13)  87 
Net operating inco


In [78]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

test_vector = embeddings.embed_query("HSBC revenue")

print("Embedding dimension:", len(test_vector))

Loading weights: 100%|█████████████| 103/103 [00:00<00:00, 534.66it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimension: 384


In [79]:
from qdrant_client import QdrantClient

client = QdrantClient(
    host="localhost",
    port=6333
)

print("Connected to Qdrant")

Connected to Qdrant


In [80]:
from langchain_community.vectorstores import Qdrant

collection_name = "annual_report_debug"

vectorstore = Qdrant.from_documents(
    chunks,
    embeddings,
    url="http://localhost:6333",
    collection_name=collection_name
)

print("Embeddings uploaded")

Embeddings uploaded


In [119]:
query = "What is the capital city of India?"

docs = vectorstore.similarity_search(query, k=5)

for i, doc in enumerate(docs):
    print("\nRESULT", i+1)
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:400])
    print("-"*50)


RESULT 1
Page: 155
India  2,344  323  667  3,334  (3)  (20)  (3)  (26) 
Indonesia  36  152  58  246  (2)  (8)  (5)  (15) 
Mainland China  5,417  164  402  5,983  (20)  (22)  (6)  (48) 
Malaysia  3,494  1,070  267  4,831  (16)  (38)  (26)  (80) 
Singapore  6,776  691  7,415  14,882  —  (38)  (36)  (74) 
Taiwan  6,570  437  1,123  8,130  —  (5)  (14)  (19) 
Egypt  —  119  278  397  —  (1)  (1)  (2) 
UAE  2,456  587  9
--------------------------------------------------

RESULT 2
Page: 154
Hong Kong  114,792  37,890  18,591  133,383  (3,515)  (1,991)  (117)  (3,632) 
Australia  14,472  4,725  4,627  19,099  (28)  (3)  (1)  (29) 
India  13,789  2,131  6,687  20,476  (53)  (5)  (7)  (60) 
Indonesia  3,063  172  573  3,636  (74)  —  (1)  (75) 
Mainland China  27,663  5,254  12,272  39,935  (281)  (197)  (4)  (285) 
Malaysia  5,560  1,059  486  6,046  (34)  (7)  —  (34) 
Singapore  16,6
--------------------------------------------------

RESULT 3
Page: 354
68 49 avenue J.F. Kennedy, Luxembour

In [120]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

context = "\n\n".join([doc.page_content for doc in docs])

prompt = f"""
You are a financial analyst.

Use only the HSBC annual report context to answer. 
You can use the context in the best way to answer the query,but if the query is really different then say that you can not answer the question if it is not from the context.

Context:
{context}

Question:
{query}

Answer:
"""

response = llm.invoke(prompt)

print(response.content)

I cannot answer the question as it is not related to the context of the HSBC annual report. The context provided does not mention the capital city of India.


In [18]:
query = "What is the salary of Georges?"

docs = vectorstore.similarity_search(query, k=5)

for i, doc in enumerate(docs):
    print("\nRESULT", i+1)
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:400])
    print("-"*50)


RESULT 1
Page: 119
5 Employee compensation and benefits
2025 2024
£m £m
Wages and salaries  1,653  1,345 
Social security costs  261  240 
Other pension costs1  69  87 
Year ended 31 Dec  1,983  1,672 
1 Includes £54m (2024: £52m) in employer contributions to the defined contribution pension plans. 
Average number of persons employed by the group during the year by global business1,2
2025 2024
Commercial and Institu
--------------------------------------------------

RESULT 2
Page: 34
5-year average1 3.6 2.2
1 The five-year average is calculated over a projected period of 20 quarters 
from 1Q26 to 4Q30.
Consensus Central scenario 2025–2029 (as at 4Q24)
UK France
GDP (annual average growth rate, %)
2025 1.2 0.9
2026 1.3 0.9
2027 1.8 1.4
2028 1.6 1.5
2029 1.6 1.4
5-year average1 1.5 1.2
Unemployment rate (%)
2025 4.9 7.5
2026 4.7 7.3
2027 4.5 7.2
2028 4.3 7.0
2029 4.3 7.0
5-year 
--------------------------------------------------

RESULT 3
Page: 123
No Director exercised share options o

In [20]:
query = "What is the average increment for 2025?"

docs = vectorstore.similarity_search(query, k=5)

for i, doc in enumerate(docs):
    print("\nRESULT", i+1)
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:400])
    print("-"*50)


RESULT 1
Page: 8
Headwinds to activity over the latter period might have reflected still-
elevated inflation and interest rates, alongside economic uncertainty 
around the time of the government’s Budget in November. 2025 also 
saw a slowdown in the labour market – the unemployment rate rose 
from 4.4% to 5.2% between January and December (ONS).
However, some headwinds to growth, including those relating to high 

--------------------------------------------------

RESULT 2
Page: 33
15% at the start of 2026. That rate has fallen in recent months to 
reflect the lowering of US tariff rates on imports from mainland China, 
the conclusion of a trade agreement with Switzerland and targeted 
tariff exemptions on key products.
In the UK forecasts have deteriorated as unemployment has risen and 
both household and business confidence has weakened.
Global GDP is expected to grow by 2
--------------------------------------------------

RESULT 3
Page: 34
5-year average1 3.6 2.2
1 The five-year 

In [21]:
query = "What is the average increment for 2025?"

docs = vectorstore.similarity_search(query, k=5)

for i, doc in enumerate(docs):
    print("\nRESULT", i+1)
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:400])
    print("-"*50)


RESULT 1
Page: 8
Headwinds to activity over the latter period might have reflected still-
elevated inflation and interest rates, alongside economic uncertainty 
around the time of the government’s Budget in November. 2025 also 
saw a slowdown in the labour market – the unemployment rate rose 
from 4.4% to 5.2% between January and December (ONS).
However, some headwinds to growth, including those relating to high 

--------------------------------------------------

RESULT 2
Page: 33
15% at the start of 2026. That rate has fallen in recent months to 
reflect the lowering of US tariff rates on imports from mainland China, 
the conclusion of a trade agreement with Switzerland and targeted 
tariff exemptions on key products.
In the UK forecasts have deteriorated as unemployment has risen and 
both household and business confidence has weakened.
Global GDP is expected to grow by 2
--------------------------------------------------

RESULT 3
Page: 34
5-year average1 3.6 2.2
1 The five-year 

In [68]:
query = "Which business unit was least profitable in 2026?"

docs = vectorstore.similarity_search(query, k=5)

for i, doc in enumerate(docs):
    print("\nRESULT", i+1)
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:400])
    print("-"*50)


RESULT 1
Page: 16
reflected notable items in 2025, including legal 
provisions of $1.4bn, restructuring and other 
related costs in 2025 of $1.0bn and $0.5bn 
related to disposals, wind-downs, acquisitions 
and related costs. 
The remaining growth in reported operating 
expenses included higher planned spend and 
investment in technology, higher performance-
related pay and the impacts of inflation. These 
increase
--------------------------------------------------

RESULT 2
Page: 23
Share of profit/(loss) from associates and joint ventures  24  45  62  (21)  (47)  — 
Profit before tax  4,367  3,969  3,212  398  10  (301) 
RoTE1 (%) 17.8 15.7 13.1
RoTE excluding notable items1 (%) 19.0 15.5 13.6
Management view of revenue – on a constant currency basis ø
2025 2024 2023 2025 vs 2024
of which strategic 
transactions2
$m $m $m $m %  $m
Banking NII3  7,000  7,640  7,288  (640)  (8
--------------------------------------------------

RESULT 3
Page: 15
Income statement results
2025 compared 

In [69]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

context = "\n\n".join([doc.page_content for doc in docs])

prompt = f"""
You are a strict financial analyst assistant that ONLY uses information from the provided HSBC annual report excerpts.

Rules you must follow exactly:
- Answer the question ONLY if it can be directly or reasonably answered using the given context.
- If the question is completely unrelated to the HSBC annual report content (for example: general knowledge questions, geography, politics, history, or anything not present in the context), respond ONLY with this exact sentence and nothing else:
  "This question is outside the scope of the HSBC annual report."


Context:
{context}

Question:
{query}

Answer:
"""

response = llm.invoke(prompt)

print(response.content)

This question is outside the scope of the HSBC annual report.


In [70]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

context = "\n\n".join([doc.page_content for doc in docs])

prompt = f"""
You are a financial analyst.

Use the HSBC annual report context to answer.

Context:
{context}

Question:
{query}

Answer:
"""

response = llm.invoke(prompt)

print(response.content)

The provided context is for the year 2025, not 2026. However, based on the given information, we can analyze the performance of different business units in 2025.

The context does not provide a direct comparison of the profitability of each business unit. However, we can look at the revenue and profit before tax for different segments.

From the income statement results, we can see that the revenue for different segments is as follows:
- Banking NII: $7,000m
- Fee and other income: $7,593m
  - Retail Banking: $665m
  - Wealth: $6,845m
  - Other: $83m

The profit before tax for the entire group is $29,907m. However, the context does not provide the profit before tax for each individual business unit.

Based on the available information, it is difficult to determine which business unit was the least profitable in 2025. The "Other" category within Fee and other income has the lowest revenue, but we cannot conclude that it is the least profitable without more information on the costs and e

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

context = "\n\n".join([doc.page_content for doc in docs])

prompt = f"""
You are a strict financial analyst assistant that ONLY uses information from the provided HSBC annual report excerpts.

Rules you must follow exactly:
- Answer the question ONLY if it can be directly or reasonably answered using the given context.
- If the question is completely unrelated to the HSBC annual report content (for example: general knowledge questions, geography, politics, history, or anything not present in the context), respond ONLY with this exact sentence and nothing else:
  "This question is outside the scope of the HSBC annual report."
- Do not add explanations, apologies, suggestions, or any additional text when declining.
- Do not attempt to guess, use outside knowledge, or be helpful beyond the given context.
- When the question is in-scope, give a concise, factual answer based only on the context.
- Use tables or numbers from the context when relevant.

Context:
{context}

Question:
{query}

Answer:
"""

response = llm.invoke(prompt)

print(response.content)

In [71]:
query = "least profitable business unit?"


In [72]:
docs = vectorstore.max_marginal_relevance_search(
    query,
    k=6,
    fetch_k=20
)

for i, doc in enumerate(docs):
    print("\nRESULT", i+1)
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:400])


RESULT 1
Page: 23
Share of profit/(loss) from associates and joint ventures  24  45  62  (21)  (47)  — 
Profit before tax  4,367  3,969  3,212  398  10  (301) 
RoTE1 (%) 17.8 15.7 13.1
RoTE excluding notable items1 (%) 19.0 15.5 13.6
Management view of revenue – on a constant currency basis ø
2025 2024 2023 2025 vs 2024
of which strategic 
transactions2
$m $m $m $m %  $m
Banking NII3  7,000  7,640  7,288  (640)  (8

RESULT 2
Page: 77
Our operations are closely integrated and, accordingly, the presentation 
of data includes internal allocations of certain items of income and 
expense. These allocations include the costs of certain support services 
and global infrastructures to the extent that they can be meaningfully 
attributed to business segments. While such allocations have been 
made on a systematic and consistent basis, 

RESULT 3
Page: 10
targeting areas of competitive strengths. The 
privatisation of Hang Seng Bank is an example 
of this. The transaction allows us to further 


In [73]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

context = "\n\n".join([doc.page_content for doc in docs])

prompt = f"""
You are a financial analyst.

Use the HSBC annual report context to answer.

Context:
{context}

Question:
{query}

Answer:
"""

response = llm.invoke(prompt)

print(response.content)

Based on the provided information, the least profitable business unit appears to be the 'Retail Banking' segment within the 'Fee and other income' category. It reported a decline in income of $100 million, representing a 13% decrease compared to the previous year.

However, if we consider the overall profitability of each business segment, the least profitable one seems to be the 'Corporate Centre' segment, which reported an operating loss of $1,597 million and a cost efficiency ratio of 55.1%. Additionally, the 'Other' segment within the 'Fee and other income' category reported an operating loss, but the amount is relatively small.

It's also worth noting that the 'Corporate Centre' segment reported a significant impairment of goodwill and other intangible assets of $405 million, which contributed to its operating loss. 

Therefore, based on the available data, the 'Corporate Centre' segment appears to be the least profitable business unit.


The return on average tangible equity in 2025 was 13.3%, or 17.2% excluding the impact of notable items.


In [13]:
questions = [
    "What are HSBC's strategic priorities?",
    "What risks does HSBC mention?",
    "What was HSBC revenue?",
    "How did HSBC perform in Asia?"
]

for q in questions:
    docs = vectorstore.max_marginal_relevance_search(q, k=5, fetch_k=20)

    print("\nQUESTION:", q)
    print("TOP PAGE:", docs[0].metadata.get("page"))


QUESTION: What are HSBC's strategic priorities?
TOP PAGE: 20

QUESTION: What risks does HSBC mention?
TOP PAGE: 34

QUESTION: What was HSBC revenue?
TOP PAGE: 126

QUESTION: How did HSBC perform in Asia?
TOP PAGE: 4
